In [38]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [8]:
file = np.load('DataTrain.npy')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #moving all the weights to the GPU
data = torch.tensor(file, dtype=torch.float32)
features = data[:,:-1] #getting just the features
fraud = data[:, -1].float() #just the fraud
dataset = torch.utils.data.TensorDataset(features, fraud) #creating a dataset with separate features and fraud
loader = torch.utils.data.DataLoader(dataset, batch_size=512, shuffle=True,num_workers=4, pin_memory=True)

In [80]:
class Model(nn.Module):
  def __init__(self, input_size, layerSize = [128, 64, 32, 16]):
    super(Model, self).__init__()
    layers = []
    c = input_size
    for s in layerSize:
        layers.append(nn.Linear(c, s))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(0.2))
        c = s
    layers.append(nn.Linear(c, 1))
    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return self.model(x)

In [82]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model(features.shape[1]).to(device)
model.load_state_dict(torch.load("model_weights2.pth", map_location=device))
model.eval()

Model(
  (model): Sequential(
    (0): Linear(in_features=33, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=32, out_features=16, bias=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=16, out_features=1, bias=True)
  )
)

In [14]:
test = np.load('DataTest.npy')
test_data = torch.tensor(test, dtype=torch.float32)
print(test_data.size(), data.size())
test_features = test_data[:,:-1] #getting just the features
test_fraud = test_data[:, -1].float() #just the fraud
test_dataset = torch.utils.data.TensorDataset(test_features, test_fraud) #creating a dataset with separate features and fraud
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=4096, shuffle=False,num_workers=4, pin_memory=True)

torch.Size([555719, 34]) torch.Size([1296675, 34])


In [47]:
total_percentage_caught=[]
total_non_frauds_blocked=[]
total_decline_rate=[]
threshold=[0.999999,0.99999,0.9999,0.999,0.998,0.997,0.996,0.995,0.994,0.993,0.992,0.991,0.99,0.989]
threshold.reverse()

In [92]:
with torch.no_grad(): #tell the model not to use the gradient cuz we are not training anymore 
    predis_sum = 0
    actual_sum = 0
    TP = 0
    total = 0
    for sample1, sample2 in test_loader: #random samples of data
        sample1=sample1.to(device)
        sample2=sample2.to(device)
        outputs=model(sample1).squeeze() #get the prediction
        probis = torch.sigmoid(outputs) #putting the prediction through the sigmoid function to get confidence score because just outputs is an array of nonsense
        predis = (probis>0.99899).float() #very basic way to find who is a fraud
        TP += ((sample2 == 1.0) & (sample2 == predis)).sum().item()
        total+=len(sample2)
        predis_sum += predis.sum().item()
        actual_sum += sample2.sum().item()
    FP = predis_sum-TP
    FN = actual_sum-TP
    TN = total-TP-FP-FN
    print("Predicted fraud:", predis_sum)
    print("Actual fraud:", actual_sum)
    print('Correct:',TP)
    print('Decline rate:',(TP+FP)/total*100,'%')
    print('False declines:',(FP)/(FP+TN)*100,'%')
    print('Fraud recall:',TP/(TP+FN)*100,'%')
    print('Precision:',FP/(FP+TP)*100,'%')
    print("\nConfusion Matrix")
    print("----------------")
    print("TP:", TP)
    print("FP:", FP)
    print("FN:", FN)
    print("TN:", TN)

Predicted fraud: 945.0
Actual fraud: 2145.0
Correct: 880
Decline rate: 0.170049971298444 %
False declines: 0.011741880940940145 %
Fraud recall: 41.02564102564102 %
Precision: 6.878306878306878 %

Confusion Matrix
----------------
TP: 880
FP: 65.0
FN: 1265.0
TN: 553509.0
